In [185]:
import pandas as pd
import numpy as np
from datetime import time
import statsmodels.api as sm
import datetime as dt
#start_date = "2002-03-01"
#end_date = "2006-03-01"
#df = pd.read_csv('data/SPY_15min_2002-01_to_2006-08.csv')
df = pd.read_csv('data/SPY_15min_2020-01_to_2022-01.csv')
start_date = "2020-01-03"
end_date = "2021-12-31"

#df['datetime'] = df['datetime'].dt.tz_localize('US/Eastern')
df['datetime'] = pd.to_datetime(df['Unnamed: 0'], utc=True)
print(df['Unnamed: 0'].head())
print(df['Unnamed: 0'].dtype)


df.set_index('datetime', inplace=True)
df.index = pd.DatetimeIndex(df.index)  # This ensures time-aware index
df = df.asfreq('15min')  # Ensure the index is at 15-minute frequency
df.index = df.index.tz_convert("Europe/Berlin")  # Convert to CET timezone
df = df.between_time("15:30", "22:00")
df = df[df.index.weekday < 5] # Keep only Monday (0) through Friday (4)

df["y^2"] = (df["close"].apply(lambda x: np.log(x)).diff()) ** 2
df = df.loc[pd.Timestamp(start_date).tz_localize("Europe/Berlin"):pd.Timestamp(end_date).tz_localize("Europe/Berlin")]

df.drop(columns=['Unnamed: 0'], inplace=True)
df.interpolate(method='time', inplace=True)


df["day_of_week"] = df.index.dayofweek
df["hour_of_day"] = df.index.hour

#todo add außerhalb der Handelszeiten dummies
day_dummies = pd.get_dummies(df["day_of_week"], prefix="day", drop_first=True)
hour_dummies = pd.get_dummies(df["hour_of_day"], prefix="hour", drop_first=True)




df = pd.concat([df, day_dummies, hour_dummies], axis=1)
df


0    2020-01-02 09:00:00
1    2020-01-02 09:15:00
2    2020-01-02 09:30:00
3    2020-01-02 09:45:00
4    2020-01-02 10:00:00
Name: Unnamed: 0, dtype: object
object


,open,high,low,close,volume,y^2,day_of_week,hour_of_day,day_1,day_2,day_3,day_4,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22
datetime,,,,,,,,,,,,,,,,,,,
2020-01-03 15:30:00+01:00,298.5430,300.2164,298.5058,300.1606,5038308.0,3.889389e-05,4,15,False,False,False,True,False,False,False,False,False,False,False
2020-01-03 15:45:00+01:00,300.1699,300.2257,299.7329,300.0351,2823734.0,1.748887e-07,4,15,False,False,False,True,False,False,False,False,False,False,False
2020-01-03 16:00:00+01:00,299.9096,300.0304,299.3704,299.9560,3244474.0,6.952217e-08,4,16,False,False,False,True,True,False,False,False,False,False,False
2020-01-03 16:15:00+01:00,299.9561,300.1606,299.5424,300.0583,1413861.0,1.162755e-07,4,16,False,False,False,True,True,False,False,False,False,False,False
2020-01-03 16:30:00+01:00,300.0676,300.3465,299.8910,300.2865,1666515.0,5.779494e-07,4,16,False,False,False,True,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-30 21:00:00+01:00,459.0583,459.0583,458.4726,458.5687,973047.0,1.138703e-06,3,21,False,False,True,False,False,False,False,False,False,True,False
2021-12-30 21:15:00+01:00,458.5687,458.7799,458.4919,458.4919,869937.0,2.805346e-08,3,21,False,False,True,False,False,False,False,False,False,True,False
2021-12-30 21:30:00+01:00,458.4967,458.6551,457.8390,457.9686,1926267.0,1.304170e-06,3,21,False,False,True,False,False,False,False,False,False,True,False


In [186]:

def safe_add(column: pd.Series, value: int): # Expects a datetimeIndex
    if not pd.api.types.is_datetime64_any_dtype(column):
        raise TypeError("Excpected a Series with DatetimeIndex.")
    # Check if the index is timezone-aware
    if column.dt.tz is None:
        raise ValueError("DatetimeIndex must be timezone-aware.")
    # Cast to ET timezone for timezone aware consistency
    column = column.dt.tz_convert("America/New_York")
    # Check whether value is outside of market open hours
    hypothetical_values = column + pd.Timedelta(minutes=value)

    mask_before_open = (hypothetical_values.dt.time < dt.time(9,30))
    mask_during_open = (dt.time(9,30) <= hypothetical_values.dt.time) & (hypothetical_values.dt.time <= dt.time(16,0))
    mask_after_open = (hypothetical_values.dt.time > dt.time(16,0))

    date_of_action = column.dt.normalize() # Extract date and set hours to 0:00
    output = column.copy()
    output[mask_during_open] = output[mask_during_open] + pd.Timedelta(minutes=value)
    if value >= 0: # Safe add
        output[mask_before_open] = date_of_action[mask_before_open] + pd.Timedelta(hours=9, minutes=30) + pd.Timedelta(minutes=value)
        output[mask_after_open] = date_of_action[mask_after_open] + pd.Timedelta(days=1) + pd.Timedelta(hours=9, minutes=30) + pd.Timedelta(minutes=value)
    else: # Safe subtract
        # Window starts when trading closes
        output[mask_before_open] = date_of_action[mask_before_open] - pd.Timedelta(days=1) + pd.Timedelta(hours=16)
        output[mask_after_open] = date_of_action[mask_after_open] + pd.Timedelta(hours=16)

    return output # Caution: returns series with ET timezone


df_news = pd.read_csv('data/WhatMovesMarkets_eventdatabase.csv')
df_news["eventstart_CET"] = pd.to_datetime(df_news["eventstart_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
df_news["eventend_CET"] = pd.to_datetime(df_news["eventend_CET"], format="%d.%m.%Y %H:%M:%S").dt.tz_localize("Etc/GMT-1")  # Ensure correct timezone
#df_news["eventstart_CET"] = df_news["eventstart_CET"].dt.floor("15min")  # Optional: Round down to the nearest 15 minutes (not mentioned in paper)
# Filter on timestamps of stock data
df_news = df_news[(df_news["eventstart_CET"] >= start_date) & (df_news["eventstart_CET"] <= end_date)]
# Correct time window creation according to paper

event_end_mask = df_news["eventend_CET"].notna()

# Only assign to rows where event_end_mask is True
df_news.loc[event_end_mask, "window_start"] = (
    safe_add(df_news.loc[event_end_mask, "eventstart_CET"], -20)
    .dt.tz_convert("Etc/GMT-1")
)

df_news.loc[event_end_mask, "window_end"] = (
    safe_add(df_news.loc[event_end_mask, "eventend_CET"], 20)
    .dt.tz_convert("Etc/GMT-1")
)

# Only assign to rows where event_end_mask is False
df_news.loc[~event_end_mask, "window_start"] = (
    safe_add(df_news.loc[~event_end_mask, "eventstart_CET"], -15)
    .dt.tz_convert("Etc/GMT-1")
)

df_news.loc[~event_end_mask, "window_end"] = (
    safe_add(df_news.loc[~event_end_mask, "eventstart_CET"], 30)
    .dt.tz_convert("Etc/GMT-1")
)


df_news


,eventstart,eventend,name,type,subtype,subsubtype,description,source,scheduled,eventstart_CET,eventend_CET,window_start,window_end
49275,02.01.2020 19:00:00,NaT,UK BRC Shop Price Index,Macro Release,UK,BRC Shop Price Index,NaN,Bloomberg,1,2020-01-03 01:00:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49276,03.01.2020 02:45:00,NaT,FR CPI and Wages,Macro Release,FR,CPI and Wages,NaN,Bloomberg,1,2020-01-03 08:45:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49277,03.01.2020 03:00:00,NaT,ES Unemployment Net,Macro Release,ES,Unemployment Net,NaN,Bloomberg,1,2020-01-03 09:00:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49278,03.01.2020 04:00:00,NaT,DE CPI Hesse,Macro Release,DE,CPI Hesse,NaN,Bloomberg,1,2020-01-03 10:00:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
49279,03.01.2020 04:30:00,NaT,UK Monetary Aggregates,Macro Release,UK,Monetary Aggregates,NaN,Bloomberg,1,2020-01-03 10:30:00+01:00,NaT,2020-01-02 22:00:00+01:00,2020-01-03 16:00:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51566,31.08.2020 11:35:00,NaT,US Auction Result Bill,Auction,US Result,Bill,CUSIP 9127964F3,https://www.treasurydirect.gov/instit/annceres...,1,2020-08-31 17:35:00+01:00,NaT,2020-08-31 17:20:00+01:00,2020-08-31 18:05:00+01:00
51567,31.08.2020 11:35:00,NaT,US Auction Result Bill,Auction,US Result,Bill,CUSIP 912796TU3,https://www.treasurydirect.gov/instit/annceres...,1,2020-08-31 17:35:00+01:00,NaT,2020-08-31 17:20:00+01:00,2020-08-31 18:05:00+01:00
51568,31.08.2020 20:00:00,NaT,IE Investec Manufacturing PMI,Macro Release,IE,Investec Manufacturing PMI,NaN,Bloomberg,1,2020-09-01 02:00:00+01:00,NaT,2020-08-31 21:00:00+01:00,2020-09-01 15:00:00+01:00
51569,31.08.2020 20:30:00,NaT,JP Markit/JMMA Manufacturing PMI,Macro Release,JP,Markit/JMMA Manufacturing PMI,NaN,Bloomberg,1,2020-09-01 02:30:00+01:00,NaT,2020-08-31 21:00:00+01:00,2020-09-01 15:00:00+01:00


In [187]:
#todo variable einfügen für Gruppierungsebene
print(f"creating {len(df_news["type"].unique())} event dummie variables")
dummies = {}
for subtype, df_type in df_news.groupby("type"):
    mask = pd.Series(False, index=df.index)
    for _, row in df_type.iterrows():
        mask = mask | ((df.index > row["window_start"]) & (df.index <= row["window_end"]))
    dummies[f"D_{subtype}"] = mask

df = pd.concat([df, pd.DataFrame(dummies)], axis=1)
df

creating 4 event dummie variables


,open,high,low,close,volume,y^2,day_of_week,hour_of_day,day_1,day_2,...,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,D_Ad Hoc,D_Auction,D_Central Bank,D_Macro Release
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-03 15:30:00+01:00,298.5430,300.2164,298.5058,300.1606,5038308.0,3.889389e-05,4,15,False,False,...,False,False,False,False,False,False,False,False,False,True
2020-01-03 15:45:00+01:00,300.1699,300.2257,299.7329,300.0351,2823734.0,1.748887e-07,4,15,False,False,...,False,False,False,False,False,False,False,False,False,True
2020-01-03 16:00:00+01:00,299.9096,300.0304,299.3704,299.9560,3244474.0,6.952217e-08,4,16,False,False,...,False,False,False,False,False,False,False,False,False,True
2020-01-03 16:15:00+01:00,299.9561,300.1606,299.5424,300.0583,1413861.0,1.162755e-07,4,16,False,False,...,False,False,False,False,False,False,False,False,False,True
2020-01-03 16:30:00+01:00,300.0676,300.3465,299.8910,300.2865,1666515.0,5.779494e-07,4,16,False,False,...,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-30 21:00:00+01:00,459.0583,459.0583,458.4726,458.5687,973047.0,1.138703e-06,3,21,False,False,...,False,False,False,False,True,False,False,False,False,False
2021-12-30 21:15:00+01:00,458.5687,458.7799,458.4919,458.4919,869937.0,2.805346e-08,3,21,False,False,...,False,False,False,False,True,False,False,False,False,False
2021-12-30 21:30:00+01:00,458.4967,458.6551,457.8390,457.9686,1926267.0,1.304170e-06,3,21,False,False,...,False,False,False,False,True,False,False,False,False,False


In [200]:
# Regression
window_amt = 4

fixed_effects = pd.concat([day_dummies, hour_dummies], axis=1).astype(float)
event_dummies = pd.DataFrame(dummies).astype(float)
lagged_variance = df['y^2'].rolling(window=window_amt).sum()  # assuming window_length 15-min intervals according to paper




#x = pd.concat([event_dummies, fixed_effects, lagged_variance], axis=1)
#x = pd.concat([event_dummies, fixed_effects, lagged_variance], axis=1).iloc[3:]
x = event_dummies.iloc[3:]
x.dropna(inplace=True) #drops first 3 rows due to rolling window
x = sm.add_constant(x)
y = df["y^2"].iloc[3:].astype(float)
#model = sm.OLS(y, x).fit(cov_type='cluster', cov_kwds={'groups': df.index.date})
model = sm.OLS(y, x).fit()
print(model.summary())
x

TypeError: merge() missing 1 required positional argument: 'right'

In [199]:
# Compute Omega_k
k = 4

rsq_full = model.rsquared
event_columns = [x for x in pd.DataFrame(dummies).columns] # Select only event dummies
tstats = model.tvalues[event_columns].abs() #
top_k_columns = tstats.nlargest(k).index # Select top k columns based on t-statistics
#print(top_k_columns)
betas = model.params[top_k_columns]

# Formula from paper
p_d_eq_1 = df[top_k_columns].mean()
product = betas * p_d_eq_1
mu_ysqr = np.mean(df["y^2"].dropna())  # Mean of y^2, excluding the first rows due to rolling window
omega_k = sum(product) / mu_ysqr
print(omega_k)

0.369644794907351


In [182]:
# Elis Formel
def compute_omega_k(model, df):
    X = df[[col for col in df.columns if col.startswith('D_')]]
    betas = model.params[X.columns] # estimated parameters --> measures the event's impact
    total_var = np.var(model.model.endog)

    omega_k = {}
    for col in X.columns:
        contribution = np.var(X[col] * betas[col]) # measures how much an eventtype impacts the variance
        omega_k[col] = max(contribution / total_var, 0) if total_var != 0 else 0
    return pd.DataFrame.from_dict(omega_k, orient='index', columns=['Omega_k']).sort_values(by='Omega_k', ascending=False)
compute_omega_k(model, df)

,Omega_k
D_Macro Release,0.000305
D_Auction,0.000256
D_Central Bank,0.000043
D_Ad Hoc,0.000004


In [ ]:
#todo time series variance decomposition
#todo bar chart für Omega_k
#todo bootstrapping
#todo overnight dummies -> difference between innerhalb der Handelszeit & außerhalb der Handelszeit plotten !
#todo plots!
#todo overall cleaning
